# 02 · Freeze the split + alignment QC (CPU, ~30–60 min)

Your training notebooks re-make the train/val/test split on every run (Step 2: `stratified_split` on
whatever Step 1 discovers). The moment new subjects are attached, every group gets reshuffled and the
test set changes — so new results could no longer be compared with the paper. This notebook:

1. Runs **your own** Step 1 discovery on all attached data (old + new).
2. Reads the paper's 43 test subjects from your old runs' `test_results_per_subject.json`.
3. Re-runs **your own** Step 2 split on the old data only and checks it reproduces those 43. If it does, the old
   train/val assignment is kept exactly; new subjects are added to train/val only.
4. Writes `split.csv`. The patched training notebooks read it instead of re-splitting.
5. **Alignment QC** (§6): checks whether T1 and tensor actually line up after Step 3's resampling, and compares
   it with the affine-aware alternative. Read the verdict before you start any training.

**Inputs to attach (Add Input):** your old `processed_adni` and `processed_hcp` datasets; the new AD
preprocessing output; the 4 HCP chunk outputs from notebook 01; the outputs of your old V2 / V4 training runs
(for `test_results_per_subject.json`); optionally the raw ADNI DICOM datasets (for scanner vendor).
Accelerator: none.

In [ ]:
!pip install -q --no-cache-dir nibabel scikit-image pydicom
import os, re, json, glob, random, warnings
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom as nd_zoom
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42   # MUST stay 42 — it is what reproduces the paper's split

# ============================ CONFIG ============================
# Substrings that identify the NEW datasets' paths under /kaggle/input (dataset or notebook names).
# Everything else is treated as the data the paper was trained on.
NEW_DATA_MARKERS = ["hcp-extension", "hcp-new", "ad-new", "adni-ad-new"]      # <- EDIT to your new dataset names
# Where your old runs' outputs are attached (searched for test_results_per_subject.json)
OLD_RESULTS_GLOB = "/kaggle/input/**/test_results_per_subject.json"
# Raw ADNI DICOM roots for the vendor lookup (None to skip)
ADNI_DICOM_ROOTS = None   # e.g. ["/kaggle/input/datasets/puravkhincha/adni-mri-dti", "/kaggle/input/datasets/puravkhincha/ad-new"]
VAL_FRAC_NEW = 0.15 / 0.90   # keeps the original 75:15 train:val ratio for new subjects
N_QC = 32                    # subjects used for the alignment QC (stratified by group)
OUT = "/kaggle/working"
# ================================================================

## 1 · Your Step 1 discovery, unchanged (finds old + new subjects)

In [ ]:
REQUIRE_TENSOR = True

HCP_TENSOR_BASE = "tensor"
ADNI_TENSOR_BASE = "tensor_lower_triangular"

SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]


def _resolve_ext(subj_dir, base_name):
    """Preprocessing output has been seen as both .nii and .nii.gz — try both,
    .nii.gz first since that's what the pipeline docs describe, .nii is what's
    actually on disk for this project. Returns the real path, or None."""
    for ext in (".nii.gz", ".nii"):
        candidate = os.path.join(subj_dir, base_name + ext)
        if os.path.exists(candidate):
            return candidate
    return None


def _file_ok(path):
    """Cheap validity check: file exists, opens, has a non-degenerate shape."""
    if path is None or not os.path.exists(path):
        return False
    try:
        img = nib.load(path)
        shape = img.shape
        if len(shape) < 3 or any(s <= 1 for s in shape[:3]):
            return False
        return True
    except Exception:
        return False


import re

def _find_processed_dirs(root_name):
    """Find every directory matching `root_name` under the search roots, tolerating
    Kaggle's automatic renaming when multiple attached datasets contain a folder with the
    same name (e.g. a second `processed_adni` upload becomes `processed_adni (1)`, and a
    second `processed_hcp` upload becomes `processed_hcp 2`). An exact-name match alone
    misses these renamed folders entirely — this happened in practice, silently dropping
    every CN subject from discovery even though the files were present on disk."""
    pattern = re.compile(rf"^{re.escape(root_name)}(\s*\(\d+\))?(\s+\d+)?$")
    hits = []
    for base in SEARCH_ROOTS:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            for d in dirs:
                if pattern.match(d):
                    hits.append(os.path.join(root, d))
    if hits:
        print(f"  {root_name}: matched folder(s) -> {[os.path.basename(h) for h in hits]}")
    return hits


def discover_subjects():
    manifest = []
    skipped = []

    # --- HCP ---
    for hcp_root in _find_processed_dirs("processed_hcp"):
        for subj in sorted(os.listdir(hcp_root)):
            subj_dir = os.path.join(hcp_root, subj)
            if not os.path.isdir(subj_dir):
                continue
            t1 = _resolve_ext(subj_dir, "T1_brain")
            fa = _resolve_ext(subj_dir, "FA")
            md_ = _resolve_ext(subj_dir, "MD")
            tensor = _resolve_ext(subj_dir, HCP_TENSOR_BASE)
            required = [t1, fa, md_] + ([tensor] if REQUIRE_TENSOR else [])
            if not all(_file_ok(p) for p in required):
                skipped.append((f"HCP:{subj}", "missing/corrupt required file(s)"))
                continue
            manifest.append(dict(subject_id=subj, group="HCP", t1=t1, fa=fa, md=md_,
                                  tensor=tensor if REQUIRE_TENSOR else None))

    # --- ADNI (AD, MCI only) ---
    for adni_root in _find_processed_dirs("processed_adni"):
        for cat in ("AD", "MCI", "CN"):
            cat_dir = os.path.join(adni_root, cat)
            if not os.path.isdir(cat_dir):
                continue
            for subj in sorted(os.listdir(cat_dir)):
                subj_dir = os.path.join(cat_dir, subj)
                if not os.path.isdir(subj_dir):
                    continue
                t1 = _resolve_ext(subj_dir, "T1_brain")
                fa = _resolve_ext(subj_dir, "FA")
                md_ = _resolve_ext(subj_dir, "MD")
                tensor = _resolve_ext(subj_dir, ADNI_TENSOR_BASE)
                required = [t1, fa, md_] + ([tensor] if REQUIRE_TENSOR else [])
                if not all(_file_ok(p) for p in required):
                    skipped.append((f"{cat}:{subj}", "missing/corrupt required file(s)"))
                    continue
                manifest.append(dict(subject_id=subj, group=cat, t1=t1, fa=fa, md=md_,
                                      tensor=tensor if REQUIRE_TENSOR else None))

    # de-duplicate by (group, subject_id) in case the same processed folder is mounted twice
    seen = set()
    deduped = []
    for m in manifest:
        key = (m["group"], m["subject_id"])
        if key in seen:
            continue
        seen.add(key)
        deduped.append(m)

    return deduped, skipped


manifest, skipped = discover_subjects()

from collections import Counter
counts = Counter(m["group"] for m in manifest)
print("Discovered usable subjects:")
for g in ("AD", "MCI", "CN", "HCP"):
    print(f"  {g}: {counts.get(g, 0)}")
print(f"Total: {len(manifest)}")

if skipped:
    print(f"\nSkipped {len(skipped)} subject(s) (missing/corrupt files):")
    for name, reason in skipped[:20]:
        print(f"  - {name}: {reason}")
    if len(skipped) > 20:
        print(f"  ... and {len(skipped)-20} more")

if len(manifest) == 0:
    raise RuntimeError(
        "No usable subjects found. Check that processed_hcp / processed_adni are attached "
        "as Kaggle Inputs or present in /kaggle/working/, and that folder names match the "
        "preprocessing notebooks' output structure."
    )

In [ ]:
def is_new(entry):
    return any(m.lower() in entry["t1"].lower() for m in NEW_DATA_MARKERS)

all_manifest = list(manifest)
n_new = sum(is_new(e) for e in all_manifest)
print(f"\nall subjects: {len(all_manifest)}   from NEW datasets: {n_new}   from OLD datasets: {len(all_manifest) - n_new}")
if n_new == 0:
    print("⚠️  No subject matched NEW_DATA_MARKERS — check the marker strings against the paths below:")
    print(sorted({os.path.dirname(os.path.dirname(e['t1'])) for e in all_manifest})[:20])

## 2 · The paper's test set (from your old runs)

In [ ]:
found = {}
for p in glob.glob(OLD_RESULTS_GLOB, recursive=True):
    rows = json.load(open(p))
    ids = frozenset((r["group"], str(r["subject_id"])) for r in rows)
    found.setdefault(ids, []).append(p)
for ids, paths in found.items():
    print(f"{len(ids):3d} test subjects in:", *paths, sep="\n    ")
if not found:
    raise RuntimeError("No test_results_per_subject.json found. Attach the output of your old V2 or V4 run.")
PAPER_TEST = max(found, key=lambda k: (len(k) == 43, len(found[k]), len(k)))
print(f"\nusing the {len(PAPER_TEST)}-subject test set (paper: 43 = 8 AD, 16 MCI, 15 CN, 4 HCP)")
print(pd.Series([g for g, _ in PAPER_TEST]).value_counts().to_dict())

## 3 · Reproduce the paper's split with your Step 2 code, then add the new subjects

In [ ]:
# your Step 2 split function, unchanged
def stratified_split(items, train_frac=0.75, val_frac=0.15, seed=SEED):
    items = items[:]
    rng = random.Random(seed)
    rng.shuffle(items)
    n = len(items)
    n_train = max(1, int(round(n * train_frac))) if n > 0 else 0
    n_val = max(1, int(round(n * val_frac))) if n - n_train > 1 else max(0, n - n_train - 1)
    n_train = min(n_train, n)
    n_val = min(n_val, n - n_train)
    train = items[:n_train]
    val = items[n_train:n_train + n_val]
    test = items[n_train + n_val:]
    return train, val, test

def run_their_split(entries):
    by_group = {"AD": [], "MCI": [], "CN": [], "HCP": []}
    for m in entries:
        by_group[m["group"]].append(m)
    out = {}
    for g, items in by_group.items():
        if not items:
            continue
        tr, va, te = stratified_split(items)
        out.update({(e["group"], e["subject_id"]): s for s, lst in (("train", tr), ("val", va), ("test", te)) for e in lst})
    return out

# old manifest = what discovery finds when the new datasets are hidden (same walk order as before)
_find_all = _find_processed_dirs
_find_processed_dirs = lambda root_name: [h for h in _find_all(root_name) if not any(m.lower() in h.lower() for m in NEW_DATA_MARKERS)]
old_manifest, _ = discover_subjects()
_find_processed_dirs = _find_all

old_assign = run_their_split(old_manifest)
repro_test = {k for k, s in old_assign.items() if s == "test"}
match = repro_test == set(PAPER_TEST)
print(f"old subjects: {len(old_manifest)}  reproduced test: {len(repro_test)}  identical to the paper's: {match}")
if not match:
    print(f"  only in reproduced: {sorted(repro_test - set(PAPER_TEST))[:6]}\n  only in paper: {sorted(set(PAPER_TEST) - repro_test)[:6]}")
    print("  -> Test set is forced to the paper's list. Train/val for old subjects uses the reproduction; any")
    print("     subject that moved is reassigned (paper-test ones go to test, others to train).")

rows = []
rng = random.Random(SEED)
key_all = {(e["group"], e["subject_id"]): e for e in all_manifest}
for k, e in key_all.items():
    if k in PAPER_TEST:
        split = "test"
    elif k in old_assign:          # an old subject (even if a copy also sits in a new dataset)
        split = old_assign[k] if old_assign[k] != "test" else "train"
    else:
        split = None   # new subject, assigned below
    rows.append(dict(group=k[0], subject_id=k[1], split=split, is_new=split is None, t1=e["t1"]))
split_df = pd.DataFrame(rows)
for g, sub in split_df[split_df.split.isna()].groupby("group"):
    ids = sorted(sub.index); rng.shuffle(ids)
    n_val = int(round(VAL_FRAC_NEW * len(ids)))
    split_df.loc[ids[:n_val], "split"] = "val"
    split_df.loc[ids[n_val:], "split"] = "train"

missing = set(PAPER_TEST) - set(zip(split_df.group, split_df.subject_id))
assert not missing, f"paper test subjects not found in the attached data: {sorted(missing)[:5]}"
assert (split_df.split == "test").sum() == len(PAPER_TEST)
assert not split_df[split_df.is_new].split.eq("test").any()

## 4 · Scanner vendor (for the per-vendor analysis, reviewer comment 4)

In [ ]:
def vendor_map(roots, ids):
    import pydicom
    ids, found = set(ids), {}
    for root in roots:
        for r, dirs, _ in os.walk(root):
            for d in [d for d in dirs if d in ids and d not in found]:
                for rr, _, files in os.walk(os.path.join(r, d)):
                    dcm = [f for f in files if f.lower().endswith(".dcm")]
                    if dcm:
                        try:
                            m = str(pydicom.dcmread(os.path.join(rr, dcm[0]), stop_before_pixels=True,
                                                    specific_tags=["Manufacturer"]).get("Manufacturer", "")).upper()
                        except Exception:
                            m = ""
                        found[d] = "GE" if "GE" in m else "Siemens" if "SIEMENS" in m else "Philips" if "PHILIPS" in m else (m or "unknown")
                        break
            dirs[:] = [d for d in dirs if d not in ids]
    return found

vendors = vendor_map(ADNI_DICOM_ROOTS, split_df.loc[split_df.group != "HCP", "subject_id"]) if ADNI_DICOM_ROOTS else {}
split_df["vendor"] = [("Siemens (HCP)" if g == "HCP" else vendors.get(s, "unknown")) for g, s in zip(split_df.group, split_df.subject_id)]
print(split_df.vendor.value_counts())

## 5 · Save `split.csv` + new Table I

In [ ]:
split_df.drop(columns=["t1"]).to_csv(os.path.join(OUT, "split.csv"), index=False)
tab = split_df.groupby(["group", "split"]).size().unstack(fill_value=0).reindex(columns=["train", "val", "test"], fill_value=0)
tab = tab.reindex(["AD", "MCI", "CN", "HCP"]).fillna(0).astype(int)
tab["Total"] = tab.sum(1); tab.loc["Total"] = tab.sum()
print(tab)
print(f"\nnew subjects: {int(split_df.is_new.sum())}  (all in train/val)  -> saved {OUT}/split.csv")

## 6 · Alignment QC — do T1 and tensor actually line up?

Three checks, on the **current** Step 3 resampling and on the **affine-aware** alternative:
* **orientation codes** of the T1 vs. diffusion files (all subjects, header only). Step 3 ignores orientation,
  so any subject whose codes differ is misaligned under the current method.
* **brain-mask Dice** between the T1 brain and the diffusion brain on the canonical grid (≥ 0.85 is good).
* **tensor-frame agreement**: correlation of each subject's off-diagonal tensor maps with the other subjects'
  average (clearly positive = consistent; ~0 = misaligned; negative = flipped axis).

In [ ]:
codes = []
for e in all_manifest:
    try:
        c_t1 = "".join(nib.aff2axcodes(nib.load(e["t1"]).affine))
        c_dw = "".join(nib.aff2axcodes(nib.load(e["fa"]).affine))
    except Exception:
        c_t1 = c_dw = "?"
    codes.append(dict(group=e["group"], subject_id=e["subject_id"], t1=c_t1, dwi=c_dw, same=c_t1 == c_dw))
codes = pd.DataFrame(codes)
print("orientation codes T1 vs diffusion, per group:")
print(codes.groupby("group").same.agg(["count", "sum", "mean"]).rename(columns={"sum": "same", "mean": "fraction_same"}))
print("\nmost common (T1, diffusion) code pairs:")
print(codes.groupby(["group", "t1", "dwi"]).size().sort_values(ascending=False).head(10))

In [ ]:
# Your Step 3 constants + resampling, unchanged (copied from the training notebook)
CANONICAL_SHAPE = (96, 112, 96)
TARGET_VOXEL_MM = 2.0
MD_SCALE = 3e-3
TENSOR_SCALE = 3e-3

def _pad_or_crop(vol, target_shape):
    pads, slices = [], []
    for cur, tgt in zip(vol.shape, target_shape):
        if cur < tgt:
            before = (tgt - cur) // 2
            pads.append((before, tgt - cur - before)); slices.append(slice(0, cur))
        else:
            pads.append((0, 0)); start = (cur - tgt) // 2; slices.append(slice(start, start + tgt))
    vol = vol[tuple(slices)]
    if any(p != (0, 0) for p in pads):
        vol = np.pad(vol, pads, mode="constant", constant_values=0)
    return vol

def _resample(volume, src_zooms, target_shape, target_voxel_mm=TARGET_VOXEL_MM, order=1):
    out = nd_zoom(volume, [sz / target_voxel_mm for sz in src_zooms], order=order)
    return _pad_or_crop(out, target_shape)

def _clean(arr):
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def harmonize_current_arrays(entry):
    def load(p):
        img = nib.load(p); return _clean(img.get_fdata(dtype=np.float32)), img.header.get_zooms()[:3]
    t1, z1 = load(entry["t1"]); fa, zf = load(entry["fa"]); md, zm = load(entry["md"]); tn, zt = load(entry["tensor"])
    t1 = _resample(t1, z1, CANONICAL_SHAPE); fa = _resample(fa, zf, CANONICAL_SHAPE); md = _resample(md, zm, CANONICAL_SHAPE)
    t1 = np.clip(t1 / (np.percentile(t1, 99.5) + 1e-8), 0, 1)
    tn = np.stack([_resample(tn[..., c], zt, CANONICAL_SHAPE) for c in range(6)], -1) / TENSOR_SCALE
    return t1, np.clip(fa, 0, 1), np.clip(md / MD_SCALE, 0, 1), tn.astype(np.float32)

# ---------------------------------------------------------------------------------------
# Affine-aware harmonisation (drop-in alternative to Step 3's zoom-only resampling)
#
# Step 3 resamples every volume with scipy zoom + centre pad/crop, using only voxel SIZE.
# It ignores each file's orientation and position (the NIfTI affine). If a subject's T1 is
# stored in a different orientation or field of view than its diffusion data (common for
# ADNI: sagittal/coronal MPRAGE vs axial DTI), the T1 and the tensor end up misaligned in
# the canonical grid, and the network is asked to learn from mismatched voxels.
#
# This version resamples T1, FA, MD and the tensor into ONE shared world-space grid
# (RAS axes, TARGET_VOXEL_MM spacing, CANONICAL_SHAPE, centred on the diffusion brain), and
# rotates the tensor frame to match. Normalisation is identical to Step 3.
# Needs from Step 3: CANONICAL_SHAPE, TARGET_VOXEL_MM, MD_SCALE, TENSOR_SCALE, _clean.
# ---------------------------------------------------------------------------------------
from scipy.ndimage import affine_transform as _affine_transform

_PAIRS = [(0, 0), (0, 1), (1, 1), (0, 2), (1, 2), (2, 2)]   # dipy order: Dxx, Dxy, Dyy, Dxz, Dyz, Dzz
FSL_BVEC_X_FLIP = True   # dipy fits tensors in the bvec frame; FSL-style bvecs flip x when det(affine) > 0


def _six_to_mat_np(t6):
    m = np.zeros(t6.shape[:-1] + (3, 3), np.float32)
    for k, (i, j) in enumerate(_PAIRS):
        m[..., i, j] = t6[..., k]
        m[..., j, i] = t6[..., k]
    return m


def _mat_to_six_np(m):
    return np.stack([m[..., i, j] for i, j in _PAIRS], -1)


def _resample_world(vol, src_aff, tgt_aff, order=1):
    M = np.linalg.inv(src_aff) @ tgt_aff                      # target voxel -> source voxel
    return _affine_transform(vol, M[:3, :3], offset=M[:3, 3], output_shape=CANONICAL_SHAPE,
                             order=order, mode="constant", cval=0.0)


def _tensor_frame_rotation(src_aff, tgt_aff):
    A = np.linalg.inv(tgt_aff[:3, :3]) @ src_aff[:3, :3]
    U, _, Vt = np.linalg.svd(A)
    R = U @ Vt
    if FSL_BVEC_X_FLIP and np.linalg.det(src_aff[:3, :3]) > 0:
        R = R @ np.diag([-1.0, 1.0, 1.0])
    return R.astype(np.float32)


def _usable_affine(img, fallback):
    """Some writers save the tensor with an identity affine; fall back to the FA/MD affine."""
    a = img.affine
    if fallback is not None and np.allclose(a[:3, :3], np.eye(3)) and not np.allclose(fallback[:3, :3], np.eye(3)):
        return fallback
    return a


def harmonize_affine_arrays(entry):
    """Returns t1, fa, md_norm, tensor_norm(H,W,D,6 or None) on the canonical grid, float32."""
    md_img = nib.load(entry["md"])
    md_raw = _clean(md_img.get_fdata(dtype=np.float32))
    com = np.array(np.nonzero(md_raw > 0)).mean(1)
    centre = (md_img.affine @ np.r_[com, 1.0])[:3]
    tgt = np.diag([TARGET_VOXEL_MM] * 3 + [1.0])
    tgt[:3, 3] = centre - TARGET_VOXEL_MM * (np.array(CANONICAL_SHAPE) - 1) / 2.0

    t1_img = nib.load(entry["t1"])
    t1 = _resample_world(_clean(t1_img.get_fdata(dtype=np.float32)), t1_img.affine, tgt)
    fa_img = nib.load(entry["fa"])
    fa = _resample_world(_clean(fa_img.get_fdata(dtype=np.float32)), fa_img.affine, tgt)
    md_r = _resample_world(md_raw, md_img.affine, tgt)

    t1_max = np.percentile(t1, 99.5)                       # same normalisation as Step 3
    t1 = np.clip(t1 / (t1_max + 1e-8), 0, 1)
    fa = np.clip(fa, 0, 1)
    md_norm = np.clip(md_r / MD_SCALE, 0, 1)

    tensor_norm = None
    if entry.get("tensor") is not None:
        tn_img = nib.load(entry["tensor"])
        tn = _clean(tn_img.get_fdata(dtype=np.float32))
        if tn.ndim == 5:
            tn = tn[..., 0, :]
        if tn.ndim != 4 or tn.shape[-1] != 6:
            raise ValueError(f"unexpected tensor shape {tn.shape}")
        taff = _usable_affine(tn_img, md_img.affine)
        comps = np.stack([_resample_world(tn[..., c], taff, tgt) for c in range(6)], -1)
        R = _tensor_frame_rotation(taff, tgt)
        comps = _mat_to_six_np(R @ _six_to_mat_np(comps) @ R.T)
        tensor_norm = (comps / TENSOR_SCALE).astype(np.float32)
    return t1.astype(np.float32), fa.astype(np.float32), md_norm.astype(np.float32), tensor_norm


# ------------------------------- QC helpers -------------------------------
def qc_mask_dice(t1, md_norm, t1_thr=0.05, md_thr=0.01):
    """Overlap of the T1 brain and the diffusion brain on the canonical grid (1.0 = perfect)."""
    a, b = t1 > t1_thr, md_norm > md_thr
    s = a.sum() + b.sum()
    return float(2 * (a & b).sum() / s) if s else float("nan")


def qc_offdiag_agreement(tensors, md_list, md_thr=0.01):
    """For each subject: correlation of its Dxy, Dxz, Dyz maps with the mean of all OTHER subjects,
    inside the shared brain. Consistent frames -> clearly positive; misaligned -> ~0;
    a flipped axis -> two of the three strongly negative."""
    offd = [1, 3, 4]   # Dxy, Dxz, Dyz in dipy order
    stack = np.stack([t[..., offd] for t in tensors])         # (N,H,W,D,3)
    masks = np.stack([m > md_thr for m in md_list])
    total = stack.sum(0)
    common = masks.mean(0) > 0.5
    out = []
    for i in range(len(tensors)):
        loo = (total - stack[i]) / max(len(tensors) - 1, 1)
        row = []
        for k in range(3):
            a, b = stack[i][..., k][common], loo[..., k][common]
            row.append(float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else float("nan"))
        out.append(row)
    return np.array(out)

In [ ]:
qc_rng = random.Random(SEED)
qc_entries = []
for g in ("AD", "MCI", "CN", "HCP"):
    pool = [e for e in all_manifest if e["group"] == g and e.get("tensor")]
    qc_entries += qc_rng.sample(pool, min(len(pool), N_QC // 4))
res, keep = [], {"current": [], "affine": []}
for i, e in enumerate(qc_entries):
    for name, fn in (("current", harmonize_current_arrays), ("affine", harmonize_affine_arrays)):
        try:
            t1, fa, md, tn = fn(e)
            keep[name].append((e, t1, fa, md, tn))
            res.append(dict(method=name, group=e["group"], subject_id=e["subject_id"], dice=qc_mask_dice(t1, md)))
        except Exception as ex:
            print(f"  {name} failed for {e['group']}:{e['subject_id']}: {ex}")
    print(f"\r{i + 1}/{len(qc_entries)}", end="")
qc = pd.DataFrame(res)
for name in ("current", "affine"):
    if len(keep[name]) > 2:
        agree = qc_offdiag_agreement([k[4] for k in keep[name]], [k[3] for k in keep[name]])
        for (e, *_), a in zip(keep[name], agree):
            m = (qc.method == name) & (qc.subject_id == e["subject_id"]) & (qc.group == e["group"])
            qc.loc[m, ["corr_Dxy", "corr_Dxz", "corr_Dyz"]] = a
qc.to_csv(os.path.join(OUT, "alignment_qc.csv"), index=False)
print("\n\nmedian per group:")
print(qc.groupby(["method", "group"])[["dice", "corr_Dxy", "corr_Dxz", "corr_Dyz"]].median().round(3))

In [ ]:
cur, aff = qc[qc.method == "current"], qc[qc.method == "affine"]
orient_bad = (~codes.same).mean()            # Step 3 cannot handle these at all: axes are permuted/flipped
bad_cur = (cur.dice < 0.85).mean()
bad_aff = (aff.dice < 0.85).mean()
neg_aff = (aff[["corr_Dxy", "corr_Dxz", "corr_Dyz"]] < 0).sum(1).ge(2).mean() if "corr_Dxy" in aff else np.nan
print(f"T1 and diffusion stored in DIFFERENT orientations: {orient_bad:.0%} of all subjects")
print(f"T1/diffusion brain Dice < 0.85 (QC sample):     current {bad_cur:.0%}   affine-aware {bad_aff:.0%}")
print(f"median Dice:                                    current {cur.dice.median():.3f}   affine-aware {aff.dice.median():.3f}")
if orient_bad > 0.05 or (bad_cur > 0.1 and bad_aff < bad_cur / 2):
    print("""
VERDICT: with the current Step 3, T1 and tensor are misaligned for a large share of subjects (Step 3 only uses
voxel size, so a T1 stored in a different orientation ends up rotated relative to its tensor).
  -> set USE_AFFINE_HARMONIZATION = True in the training patch (notebook 03) and retrain every variant you report.
     This is very likely a major cause of the low directional accuracy and FA SSIM.""")
elif bad_cur <= 0.1:
    print("\nVERDICT: current Step 3 alignment looks fine. Keep USE_AFFINE_HARMONIZATION = False.")
else:
    print("\nVERDICT: unclear — look at the figure below and alignment_qc.csv, and send me the printout.")
if neg_aff == neg_aff and neg_aff > 0.1:
    print(f"\nNOTE: {neg_aff:.0%} of subjects have a flipped tensor frame under the affine method. "
          "Try FSL_BVEC_X_FLIP = False (in the affine cell) and re-run this QC; keep whichever gives positive correlations.")

In [ ]:
# picture: worst 3 subjects under the current method, current vs affine. Red = diffusion-brain outline.
worst = cur.sort_values("dice").head(3)
fig, axs = plt.subplots(len(worst), 2, figsize=(7, 3.4 * len(worst)), squeeze=False)
for r, (_, w) in enumerate(worst.iterrows()):
    for c, name in enumerate(("current", "affine")):
        hit = [k for k in keep[name] if k[0]["subject_id"] == w.subject_id and k[0]["group"] == w.group]
        if not hit:
            continue
        _, t1, fa, md, _ = hit[0]
        z = t1.shape[2] // 2
        axs[r, c].imshow(np.rot90(t1[:, :, z]), cmap="gray")
        axs[r, c].contour(np.rot90(md[:, :, z] > 0.01), levels=[0.5], colors="r", linewidths=0.8)
        d = qc[(qc.method == name) & (qc.subject_id == w.subject_id)].dice.iloc[0]
        axs[r, c].set_title(f"{w.group}:{w.subject_id}  {name}  Dice={d:.2f}", fontsize=8); axs[r, c].axis("off")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "alignment_qc.png"), dpi=130); plt.show()